# 00C Demo Setup — reset and prepare Day 1

Load the existing `00_env_config`, run the single reset cell, then explore the I/O examples. Start with `RESET_DEMO = False` to preview. Set it to `True` only for the dedicated, manually created `demo_metadata` lakehouse and reserved `demo` business tables.


## 1. Load the shared Fabric configuration

The setup notebook uses the same workspace-local `00_env_config` as the rest of the Guided Demo.


In [1]:
from fabricops_kit import setup_notebook

### Use this when the 00_env_config notebook is in the same workspace as this notebook
context = setup_notebook()

### Advanced you can run the 00_env_config notebook that is in another workspace, 
### make sure you have workspace execution rights to that workspace for this to work
### The caller workspace is detected automatically. Configuration workspace example: "<configuration-workspace-id>"
# Advanced: load 00_env_config from another workspace:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


## 2. Import the FabricOps I/O helpers


In [ ]:
from fabricops_kit import (
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_json,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
)


## Reset first, then explore the I/O examples

Stop all demo jobs. `False` previews; `True` clears the dedicated `demo_metadata` tables and saved lifecycle artifacts, removes six outputs and restores five Day 1 sources. Contracts, activations and Run history are discarded. Use only the manually created demo item selected by `00_env_config`. The cell checks non-demo references and active jobs before deletion. Wait for `RESET COMPLETE`.


In [ ]:
RESET_DEMO = False  # False: preview only. True: discard ALL dedicated demo metadata and reset demo data.

# Demo-only validation and physical cleanup using existing FabricOps foundations.
import json
from urllib.error import HTTPError

from delta.tables import DeltaTable
from pyspark.sql import functions as F

from fabricops_kit import resolve_table_id
from fabricops_kit.activated_pipeline.shared import FabricNotebookClient, snapshot_file
from fabricops_kit.config.metadata_schemas import metadata_table_physical_schema, metadata_table_schema_registry
from fabricops_kit.config.shared import resolve_fabric_context
from fabricops_kit.io.shared import execute_warehouse_sql, resolve_configured_lakehouse_table, resolve_warehouse_name

_DEMO_SOURCES = {
    ('Bronze', 'orders'): ('demo/orders.csv', 120),
    ('Bronze', 'products'): ('demo/products.csv', 8),
    ('Gold', 'order_history'): ('demo/order_history.csv', 180),
    ('Bronze', 'inventory_movements'): ('demo/inventory_day1.csv', 8),
    ('Bronze', 'product_master_updates'): ('demo/products_day1.csv', 8),
}
_DEMO_OUTPUTS = (
    ('Silver', 'curated_orders'), ('Gold', 'customer_summary'),
    ('Silver', 'inventory_movements_append_all'), ('Silver', 'inventory_movements_append_new_only'),
    ('Silver', 'product_master_scd1'), ('Silver', 'product_master_scd2'),
)
_DEMO_NAMES = {'02A_full_refresh_demo', '02B_incremental_append_demo', '02C_scd_demo'}


def _demo_metadata_counts():
    """Read every installed metadata schema without modifying its records or history."""
    counts = {}
    for table, expected in metadata_table_schema_registry().items():
        schema = metadata_table_physical_schema(_demo_config, table)
        try:
            frame = read_lakehouse_table(table, store='Metadata', schema=schema)
        except Exception as exc:
            raise RuntimeError(f'Cannot read {schema}.{table}. Check the demo Metadata mapping and run metadata setup in 00B.') from exc
        actual = {field.name: field.dataType.simpleString() for field in frame.schema}
        incompatible = [field.name for field in expected if actual.get(field.name) != field.dataType.simpleString()]
        if incompatible:
            raise RuntimeError(f'{table} has missing columns or incompatible types: {", ".join(incompatible)}. '
                               'Run the installed package metadata setup in the demo lakehouse, then restart 00C.')
        counts[table] = frame.count()
    return counts


def _demo_assert_source(actual, expected):
    """Compare full row multisets after normalizing Warehouse connector data types."""
    actual = actual.select(*[F.col(field.name).cast(field.dataType).alias(field.name) for field in expected.schema])
    if actual.exceptAll(expected).limit(1).count() or expected.exceptAll(actual).limit(1).count():
        raise RuntimeError('Day 1 source verification failed: persisted content differs from the fixture. Rerun 00C.')


def _demo_check_jobs():
    """Check current and retained promoted demo notebooks before resetting."""
    jobs = set()
    page = f'/workspaces/{context.workspace_id}/notebooks'
    while page:
        payload = _demo_client.request('GET', page)
        jobs.update((context.workspace_id, item['id']) for item in payload['value'] if item['displayName'] in _DEMO_NAMES)
        page = payload.get('continuationUri')
    for row in read_lakehouse_table('METADATA_PIPELINE_REGISTRY', store='Metadata',
            schema=metadata_table_physical_schema(_demo_config, 'METADATA_PIPELINE_REGISTRY')).collect():
        jobs.add((row['source_workspace_id'], row['source_notebook_id']))
        jobs.add((row['execution_workspace_id'], row['governed_notebook_id']))
    for row in read_lakehouse_table('METADATA_PIPELINE_RUN', store='Metadata',
            schema=metadata_table_physical_schema(_demo_config, 'METADATA_PIPELINE_RUN')).select('_workspace_id', '_notebook_id').distinct().collect():
        jobs.add((row['_workspace_id'], row['_notebook_id']))
    jobs.discard((context.workspace_id, _demo_runtime.get('notebook_id')))
    for workspace, notebook in jobs:
        path = f'/workspaces/{workspace}/items/{notebook}/jobs/instances'
        while path:
            try:
                payload = _demo_client.request('GET', path)
            except HTTPError as exc:
                if exc.code != 404:
                    raise
                try:
                    _demo_client.request('GET', f'/workspaces/{workspace}/items/{notebook}')
                except HTTPError as missing:
                    error = json.loads(missing.read()) if missing.code == 404 else {}
                    if missing.code == 404 and error.get('errorCode') == 'ItemNotFound':
                        break
                    raise
                raise RuntimeError('Cannot check demo notebook jobs. Stop executions and check Fabric permissions.') from exc
            if any(job['status'] not in {'Completed', 'Failed', 'Cancelled', 'Deduped'} for job in payload['value']):
                raise RuntimeError('A demo notebook job is active. Stop demo executions before RESET_DEMO=True.')
            path = payload.get('continuationUri')


def _demo_check_metadata_scope(value):
    """Reject references outside the configured demo before clearing its dedicated metadata."""
    if isinstance(value, list):
        for item in value:
            _demo_check_metadata_scope(item)
    elif isinstance(value, dict):
        for key, item in value.items():
            if key in {'table_id', 'source_table_id', 'target_table_id', 'incremental_target_table_id'}:
                if item and item not in _demo_table_ids:
                    raise RuntimeError(f'Non-demo table reference in {key}; metadata is preserved.')
            elif key == 'source_table_ids' and item:
                if not set(item).issubset(_demo_table_ids):
                    raise RuntimeError('Non-demo source references; metadata is preserved.')
            elif key in {'workspace_id', '_workspace_id', 'execution_workspace_id', 'source_workspace_id', 'evidence_workspace_id'}:
                if item and str(item).casefold() not in _demo_workspaces:
                    raise RuntimeError('Non-demo workspace reference; metadata is preserved.')
            elif key in {'operations', 'profiling', 'operations_json', 'package_json', 'contract_payload_json', 'destination_json'}:
                _demo_check_metadata_scope(json.loads(item) if isinstance(item, str) else item)
            elif isinstance(item, (dict, list)):
                _demo_check_metadata_scope(item)


def _demo_metadata_files(*, delete=False):
    """Inspect or clear only the three native lifecycle artifact folders in demo Metadata."""
    folders = {'submissions': '.json', 'dry_runs': '.json', 'pipeline_snapshots': '.ipynb'}
    counts = {}
    for folder, suffix in folders.items():
        uri = _metadata.root + '/Files/fabricops/engineering/' + folder
        path = spark._jvm.org.apache.hadoop.fs.Path(uri)
        fs = path.getFileSystem(spark._jsc.hadoopConfiguration())
        counts[folder] = 0
        if not fs.exists(path):
            continue
        files = fs.listFiles(path, True)
        while files.hasNext():
            file = files.next().getPath()
            if not str(file.getName()).endswith(suffix):
                raise RuntimeError(f'Unexpected content in {uri}; metadata is preserved. Inspect it before resetting.')
            payload = json.loads(snapshot_file(str(file), spark_session=spark))
            _demo_check_metadata_scope(payload)
            counts[folder] += 1
        if delete and (not fs.delete(path, True) or fs.exists(path)):
            raise RuntimeError(f'Could not clear {uri}. Check permissions and rerun the reset cell.')
    return counts


def _demo_remove_outputs(*, preview=False):
    """Remove exact configured output objects and their physical publication markers."""
    import notebookutils
    for store, table in _DEMO_OUTPUTS:
        if store == 'Gold':
            if preview:
                continue
            gold = context.paths[store]
            execute_warehouse_sql(spark, gold,
                "IF OBJECT_ID(N'demo.customer_summary', N'U') IS NOT NULL DROP TABLE [demo].[customer_summary]; "
                'SELECT 1 AS demo_reset_dropped;', database_name=resolve_warehouse_name(gold, store))
        else:
            _, _, _, path = resolve_configured_lakehouse_table(store, table, 'demo')
            qualified = f'{_demo_silver_catalog}.`{table}`'
            if spark.catalog.tableExists(qualified):
                location_rows = spark.sql(f'DESCRIBE TABLE EXTENDED {qualified}').where("col_name = 'Location'").collect()
                if len(location_rows) != 1 or location_rows[0]['data_type'].rstrip('/') != path.rstrip('/'):
                    raise RuntimeError(f'Catalog location mismatch for {store}.demo.{table}; reset blocked.')
                if not preview:
                    spark.sql(f'DROP TABLE {qualified}')
            if not preview and notebookutils.fs.exists(path) and not notebookutils.fs.rm(path, True):
                raise RuntimeError(f'Could not remove {store}.demo.{table}; check permissions and retry 00C.')
        if not preview:
            print(f'Removed output (missing is OK): {store}.demo.{table}')


def _demo_verify_outputs():
    """Fail if any of the six physical outputs still exists."""
    import notebookutils
    for store, table in _DEMO_OUTPUTS:
        if store == 'Gold':
            exists = read_warehouse_query(
                "SELECT COUNT(*) AS n FROM sys.tables t JOIN sys.schemas s ON t.schema_id=s.schema_id "
                "WHERE s.name='demo' AND t.name='customer_summary'", store='Gold').first()['n'] != 0
        else:
            _, _, _, path = resolve_configured_lakehouse_table(store, table, 'demo')
            exists = notebookutils.fs.exists(path) or spark.catalog.tableExists(f'{_demo_silver_catalog}.`{table}`')
        if exists:
            raise RuntimeError(f'Reset incomplete: {store}.demo.{table} remains. Check permissions and rerun 00C.')

_demo_config, _, _demo_runtime = resolve_fabric_context()
if type(RESET_DEMO) is not bool:
    raise ValueError('RESET_DEMO must be True or False.')
if context.readiness_status != 'ready':
    raise RuntimeError('Correct 00_env_config before preparing the demo.')
_metadata = context.paths['Metadata']
if _metadata.kind != 'lakehouse' or not _metadata.schema_enabled:
    raise RuntimeError('Metadata must be a schema-enabled Lakehouse.')
for _key in ('Metadata', 'Bronze', 'Silver', 'Gold'):
    _store = context.paths[_key]
    print(f'{_key}: {_store.name} | workspace {_store.workspace_id} | item {_store.item_id}')
_demo_counts = _demo_metadata_counts()
for _table, _count in _demo_counts.items():
    print(f'{_table}: {_count} rows')
for _store, _table in _DEMO_SOURCES:
    print(f'Restore Day 1: {_store}.demo.{_table}')
for _store, _table in _DEMO_OUTPUTS:
    print(f'Remove output: {_store}.demo.{_table}')

if not RESET_DEMO:
    print('Preview only. Stop demo executions and set RESET_DEMO=True to reset.')
else:
    _demo_client = FabricNotebookClient()
    _item = _demo_client.request('GET', f'/workspaces/{_metadata.workspace_id}/items/{_metadata.item_id}')
    if _item['displayName'] != 'demo_metadata' or _metadata.name != 'demo_metadata':
        raise RuntimeError('Reset requires the manually created dedicated demo_metadata item selected in 00_env_config.')
    if any((context.paths[key].workspace_id, context.paths[key].item_id) == (_metadata.workspace_id, _metadata.item_id)
           for key in ('Bronze', 'Silver', 'Gold')):
        raise RuntimeError('Metadata must be separate from the business stores.')
    _demo_table_ids = {resolve_table_id(store=store, schema='demo', table_name=table,
                       context={'config': _demo_config, 'scope_workspace_id': workspace})
                       for workspace, stores in _demo_config.path_config.paths.items()
                       for store, table in [*_DEMO_SOURCES, *_DEMO_OUTPUTS] if store in stores}
    _demo_workspaces = {workspace.casefold() for workspace in _demo_config.path_config.paths}
    _demo_workspaces.update(store.workspace_id.casefold() for store in context.paths.values())
    _demo_order = ('METADATA_GUARDRAIL_RESULTS', 'METADATA_PROCESSING_STATE', 'METADATA_DATA_LINEAGE',
        'METADATA_DATA_ACCESS', 'METADATA_PIPELINE_RUN', 'METADATA_PIPELINE_REGISTRY',
        'METADATA_PIPELINE_ACTIVATION', 'METADATA_DATA_CONTRACT_ACTIVATION', 'METADATA_DATA_PROFILED_FREQUENCY',
        'METADATA_DATA_PROFILED', 'METADATA_DATA_CONTRACT', 'METADATA_DATA_CATALOGUE',
        'METADATA_DATA_AGREEMENT', 'METADATA_DATA_STEWARD')
    if set(_demo_counts) != set(_demo_order):
        raise RuntimeError('Metadata inventory changed. Review reset dependencies before deleting records.')
    for _table in _demo_counts:
        for _row in read_lakehouse_table(_table, store='Metadata',
                schema=metadata_table_physical_schema(_demo_config, _table)).toLocalIterator():
            _demo_check_metadata_scope(_row.asDict(recursive=True))
    _demo_file_counts = _demo_metadata_files()
    for _folder, _count in _demo_file_counts.items():
        print(f'Metadata Files/{_folder}: {_count} artifacts')
    _demo_check_jobs()
    _silver = context.paths['Silver']
    _workspace_name = _demo_client.request('GET', f'/workspaces/{_silver.workspace_id}')['displayName']
    _item_name = _demo_client.request('GET', f'/workspaces/{_silver.workspace_id}/items/{_silver.item_id}')['displayName']
    _demo_silver_catalog = '.'.join('`' + name.replace('`', '``') + '`' for name in (_workspace_name, _item_name, 'demo'))
    _demo_day1 = {}
    for (_store, _table), (_file, _count) in _DEMO_SOURCES.items():
        _frame = read_lakehouse_csv(_file, store='Bronze', header=True, inferSchema=True).cache()
        if _frame.count() != _count:
            raise RuntimeError(f'{_file} must contain {_count} rows. Restore the fixture before resetting.')
        _demo_day1[(_store, _table)] = _frame
    _demo_remove_outputs(preview=True)
    # Checks and all five fixtures are complete before any destructive operation.
    for _table in _demo_order:
        _, _, _, _path = resolve_configured_lakehouse_table('Metadata', _table,
            metadata_table_physical_schema(_demo_config, _table))
        _removed = 0
        if _demo_counts[_table]:
            _delta = DeltaTable.forPath(spark, _path)
            _delta.delete()
            _removed = int(_delta.history(1).first()['operationMetrics']['numDeletedRows'])
        print(f'{_table}: removed {_removed}')
    if any(_demo_metadata_counts().values()):
        raise RuntimeError('Metadata reset incomplete. Stop all demo writers and rerun the reset cell.')
    _demo_removed_files = _demo_metadata_files(delete=True)
    for _folder, _count in _demo_removed_files.items():
        print(f'Metadata Files/{_folder}: removed {_count}')
    _demo_remove_outputs()
    for (_store, _table), _expected in _demo_day1.items():
        _writer = write_warehouse_table if _store == 'Gold' else write_lakehouse_table
        _reader = read_warehouse_table if _store == 'Gold' else read_lakehouse_table
        _writer(_expected, table_name=_table, store=_store, schema='demo', mode='overwrite')
        _demo_assert_source(_reader(table_name=_table, store=_store, schema='demo'), _expected)
    _demo_verify_outputs()
    if any(_demo_metadata_counts().values()):
        raise RuntimeError('Metadata changed during preparation. Stop demo writers and rerun the reset cell.')
    if any(_demo_metadata_files().values()):
        raise RuntimeError('Lifecycle artifacts remain. Stop demo writers and rerun the reset cell.')
    for _frame in _demo_day1.values():
        _frame.unpersist()
    print('RESET COMPLETE')


## 3. Read the same "Orders" dataset via CSV, JSON, Parquet, and Excel.

All four files contain the same 120 logical Orders rows. Paths are relative to the configured Lakehouse `Files` area, so `demo/orders.csv` resolves under `Files/demo/orders.csv`.


In [ ]:
orders_csv_df = read_lakehouse_csv(
    "demo/orders.csv",
    store="Bronze",
    
    header=True,
    inferSchema=True,
)

#display(orders_csv_df)


In [ ]:
orders_json_df = read_lakehouse_json(
    "demo/orders.json",
    store="Bronze",
    
)
#display(orders_json_df)


In [ ]:
orders_parquet_df = read_lakehouse_parquet(
    "demo/orders.parquet",
    store="Bronze",
    
)

#display(orders_parquet_df)


In [ ]:
orders_excel_df = read_lakehouse_excel(
    "demo/orders.xlsx",
    store="Bronze",
    
)
#display(orders_excel_df)


## 4. Read the restored Lakehouse and Warehouse sources

The reset cell wrote these tables with the public write helpers. The following cells demonstrate the public readers.


In [ ]:
# Sources were written and verified in the reset cell. Inspect with the public reader.
if RESET_DEMO:
    orders_table_df = read_lakehouse_table(table_name="orders", store="Bronze", schema="demo")


In [ ]:
if RESET_DEMO:
    products_table_df = read_lakehouse_table(table_name="products", store="Bronze", schema="demo")


In [ ]:
if RESET_DEMO:
    order_history_table_df = read_warehouse_table(table_name="order_history", store="Gold", schema="demo")
    order_history_sample_df = read_warehouse_query("SELECT TOP 5 * FROM demo.order_history ORDER BY 1", store="Gold")


## 5. Inspect Day 1 append and SCD sources

The reset cell already restored both sources. Do not reset between scenario days.


In [ ]:
if RESET_DEMO:
    for table in ("inventory_movements", "product_master_updates"):
        print(f"Bronze demo.{table}:", read_lakehouse_table(table, store="Bronze", schema="demo").count(), "rows")
